# 🌊 SST Amérique Centrale 2024–2025
### Données Copernicus Marine — OSTIA L4 NRT

**Dataset** : `METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2`  
**Région** : Amérique Centrale + Caraïbes + Pacifique Est tropical  
**Résolution** : 0.05° × 0.05° (≈ 5 km), journalière  

---
> **Prérequis** : compte gratuit sur [marine.copernicus.eu](https://marine.copernicus.eu)  
> **Connexion** : lancer une fois en terminal `copernicusmarine login`


## 1. Imports

In [ ]:
import copernicusmarine
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

OUT_DIR = Path("output_sst")
OUT_DIR.mkdir(exist_ok=True)

print("✓ Imports OK")

## 2. Paramètres

In [ ]:
# ── Région Amérique Centrale ──────────────────────────────────────
LON_MIN, LON_MAX = -100, -60
LAT_MIN, LAT_MAX =    5,  30

# ── Années d'analyse ──────────────────────────────────────────────
START_YEAR = 2023
END_YEAR   = 2024
YEARS      = list(range(START_YEAR, END_YEAR + 1))

# ── Dataset Copernicus ─────────────────────────────────────────────
#DATASET_ID = "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2"
DATASET_ID = "METOFFICE-GLO-SST-L4-REP-OBS-SST"
VARIABLE   = "analysed_sst"

print(f"Région  : lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"Années  : {START_YEAR} → {END_YEAR}")
print(f"Dataset : {DATASET_ID}")

## 3b. Téléchargement et sauvegarde locale (NetCDF)
Télécharge les données brutes et les sauvegarde en fichiers **NetCDF** par année sur disque.  
À n'exécuter qu'une seule fois — les cellules suivantes rechargent depuis le disque si les fichiers existent déjà.

In [ ]:
from pathlib import Path
import xarray as xr

DATA_DIR = Path("/data0/user/gcambon/DATA/DATASETS_PSF/CAMERICA_DS/")

ds_list = [xr.open_dataset(DATA_DIR / f"sst_central_america_{y}.nc") for y in YEARS]
ds_local = xr.concat(ds_list, dim="time")

sst = ds_local[VARIABLE] - 273.15
sst.attrs["units"]     = "°C"
sst.attrs["long_name"] = "Sea Surface Temperature"

print(f"Dimensions chargées : {dict(sst.sizes)}")
sst

## 4. Calcul des moyennes mensuelles

In [ ]:
sst_monthly = sst.resample(time="ME").mean()
print(f"Nombre de mois : {len(sst_monthly.time)}")
sst_monthly

## 5. Fonction de cartographie

In [ ]:
def base_map(ax, title=""):
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle="--", zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                      color="gray", alpha=0.5, linestyle="--")
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlocator = mticker.FixedLocator(range(-100, -55, 10))
    gl.ylocator = mticker.FixedLocator(range(5, 35, 5))
    if title:
        ax.set_title(title, fontsize=10, pad=6)
    return ax

print("✓ Fonction base_map définie")

## 6. Cartes mensuelles 2023

In [ ]:
year = START_YEAR
sst_yr = sst_monthly.sel(time=sst_monthly.time.dt.year == year)
n_months = len(sst_yr.time)

ncols = 4
nrows = int(np.ceil(n_months / ncols))
#vmin, vmax = float(sst_yr.min()), float(sst_yr.max())
vmin, vmax = (15,32)
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 4, nrows * 3.2),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

for i, t in enumerate(sst_yr.time.values):
    im = sst_yr.isel(time=i).plot(
        ax=axes[i], transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax,
        add_colorbar=False, add_labels=False,
    )
    base_map(axes[i], title=str(t)[:7])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.subplots_adjust(right=0.88, top=0.92, wspace=0.18, hspace=0.28)
cbar_ax = fig.add_axes([0.905, 0.18, 0.02, 0.62])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle(f"SST mensuelle — Amérique Centrale {year}", fontsize=14, y=0.98)
fig.savefig(OUT_DIR / f"sst_monthly_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Cartes mensuelles 2024

In [ ]:
year = END_YEAR
sst_yr = sst_monthly.sel(time=sst_monthly.time.dt.year == year)
n_months = len(sst_yr.time)

ncols = 4
nrows = int(np.ceil(n_months / ncols))
#vmin, vmax = float(sst_yr.min()), float(sst_yr.max())
vmin, vmax = (15,32)

nrows
fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 4, nrows * 3.2),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

for i, t in enumerate(sst_yr.time.values):
    im = sst_yr.isel(time=i).plot(
        ax=axes[i], transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax,
        add_colorbar=False, add_labels=False,
    )
    base_map(axes[i], title=str(t)[:7])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.subplots_adjust(right=0.88, top=0.92, wspace=0.18, hspace=0.28)
cbar_ax = fig.add_axes([0.905, 0.18, 0.02, 0.62])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle(f"SST mensuelle — Amérique Centrale {year}", fontsize=14, y=0.98)
fig.savefig(OUT_DIR / f"sst_monthly_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Comparaison annuelle 2023 vs 2024

In [ ]:
annual_means = {}

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

for ax, year in zip(axes, [START_YEAR, END_YEAR]):
    sst_yr = sst.sel(time=sst.time.dt.year == year).mean("time")
    annual_means[year] = sst_yr
    im = sst_yr.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=24, vmax=32,
        add_colorbar=False, add_labels=False,
    )
    base_map(ax, title=f"Moyenne annuelle {year}")

fig.subplots_adjust(top=0.98, bottom=0.1, wspace=0.1)
cbar_ax = fig.add_axes([0.22, 0.08, 0.56, 0.035])
cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle("SST annuelle — Amérique Centrale", fontsize=14, y=0.98)
fig.savefig(OUT_DIR / "sst_annual_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Anomalie SST (2024 − 2023)

In [ ]:
anomalie = annual_means[END_YEAR] - annual_means[START_YEAR]
lim = float(max(abs(anomalie.min()), abs(anomalie.max())))

fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
im = anomalie.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="RdBu_r", vmin=-lim, vmax=lim,
    add_colorbar=False, add_labels=False,
)
base_map(ax, title=f"Anomalie SST ({END_YEAR} − {START_YEAR})")
cbar = fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.8, pad=0.04)
cbar.set_label("ΔT (°C)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "sst_anomaly_2024_minus_2023.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Série temporelle (moyenne spatiale)

In [ ]:
sst_ts = sst.mean(dim=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(14, 4))
for year, color, lw in [(START_YEAR, "#2166ac", 1.4), (END_YEAR, "#d6604d", 1.4)]:
    ts = sst_ts.sel(time=sst_ts.time.dt.year == year)
    ax.plot(ts.time.values, ts.values, label=str(year), color=color, linewidth=lw)

ax.set_ylabel("SST moyenne (°C)", fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.set_title("Série temporelle SST — Amérique Centrale (moyenne spatiale)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "sst_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Fichiers générés

In [ ]:
for f in sorted(OUT_DIR.glob("*.png")):
    print(f"✓ {f}")